In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
df_census = pd.read_csv(
    'population/co-est2021-alldata.csv',
    header=0,
    usecols=['STNAME','CTYNAME','POPESTIMATE2021','STATE','COUNTY'],
    encoding='latin-1'
)

df_counties = pd.read_excel(
    'population/county_reeds_corrected0310.xlsx',
    header=0,
    usecols=['NAME','STATE_NAME','FIPS','PCA_REG']
)
df_hierarchy = pd.read_csv("hierarchy.csv")

df_census['STATE'] = df_census['STATE'].apply(lambda x: '{0:0>2}'.format(x))
df_census['COUNTY'] = df_census['COUNTY'].apply(lambda x: '{0:0>3}'.format(x))
df_census['FIPS'] = df_census['STATE'] + df_census['COUNTY']
df_census['FIPS'] = df_census['FIPS'].apply(pd.to_numeric)

df_census_counties = pd.merge(df_census, df_counties, on='FIPS', how='inner')
df_census_counties['county_state'] = df_census_counties['CTYNAME'] + ', ' + df_census_counties['STNAME']
df_census_counties['FIPS'] = 'p' + df_census_counties['FIPS'].astype(str).str.zfill(5)

In [3]:
df_19 = pd.read_excel('population/co-est2019-annres.xlsx')
df_19.columns = df_19.loc[2]
df_19 = df_19[4:].set_index(np.nan)[range(2016, 2020)]
df_19.columns.names = ['']
df_19.index = [county[1:] if county.startswith('.') else county for county in df_19.index]
df_19 = df_19.loc[df_19.index.isin(df_census_counties['county_state'])]

df_23 = pd.read_excel('population/co-est2023-pop.xlsx')
df_23.columns = df_23.loc[2]
df_23 = df_23[4:].set_index(np.nan)[range(2020, 2024)]
df_23.columns.names = ['']
df_23.index = [county[1:] if county.startswith('.') else county for (county, _) in df_23.index]
df_23 = df_23.loc[df_23.index.isin(df_census_counties['county_state'])]

In [4]:
county_populations = (
    pd.concat([df_19, df_23], axis=1)
    .merge(
        df_census_counties.set_index('county_state')[['FIPS']],
        left_index=True,
        right_index=True,
        how='left'
    )
    .set_index('FIPS')
)
county_populations.columns = [int(col) for col in county_populations.columns]

In [5]:
## CT county populations
df = pd.read_excel('population/connecticut/ct_cou_to_cousub_crosswalk.xlsx')
df = df.loc[(df.COUSUBFP.notna()) & (df.COUSUBFP != 0)].copy().reset_index(drop=True)
df['FIPS'] = (
    'p'
    + (df['STATEFP\n(INCITS38)']
    + df['NEW_COUNTYFP\n(INCITS31)'].astype(int).astype(str)).str.zfill(5)
)
df = df[['COUSUB_NAMELSAD', 'OLD_COUNTY_NAMELSAD', 'FIPS']]
df.columns = ['town', 'county', 'FIPS']
df['Town'] = df['town'].str.replace(' town', '')
df['County'] = df['county'].str.replace(' County', '')

ct_county_pops_by_year = {}
ct_town_pops_by_year = {}
for year in range(2016, 2024):
    if year < 2022:
        town_pops = pd.read_excel(f"population/connecticut/pop_towns{year}.xlsx")
        if year == 2021:
            town_pops = town_pops.loc[8:51]
        else:
            town_pops = town_pops.loc[9:53]
        town_pops.columns = town_pops.iloc[0]
        town_pops = town_pops.iloc[1:]
        
        town_pops = (
            pd.concat([
                town_pops.iloc[:, :2],
                town_pops.iloc[:, 2:4],
                town_pops.iloc[:, 4:6],
                town_pops.iloc[:, 6:8]
            ])
            .dropna()
            .reset_index(drop=True)
        )
        town_pops.columns = ['Town', 'population']
    else:
        town_pops = pd.read_excel(f"connecticut/pop_towns{year}.xlsx")
        town_pops = town_pops.iloc[:, :2].loc[16:184].copy().reset_index(drop=True)
        town_pops.columns = ['Town', 'population']
        town_pops['Town'] = town_pops['Town'].str.title()
    
    df_new = df.merge(town_pops, on='Town')
    assert len(df_new) == 169

    ct_town_pops_by_year[year] = (
        df_new
        .rename(columns={'FIPS': 'planning_region_FIPS'})
        .set_index(['Town', 'County', 'planning_region_FIPS'])
        ['population']
    )
    ct_county_pops_by_year[year] = df_new.groupby('FIPS')['population'].sum()

ct_county_populations = pd.concat(ct_county_pops_by_year, axis=1)

In [6]:
county_populations = pd.concat([county_populations, ct_county_populations.astype(float)])

In [9]:
os.makedirs("population/processed", exist_ok=True)
county_populations.to_csv("population/processed/county_populations_by_year.csv")